In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

sns.set_theme(style="whitegrid")

TARGET_COLUMN = "target"
METRIC = "f1"

print("Environment siap. Siap untuk membandingkan model.")

## 1. Impor Library dan Setup

Pada bagian ini kita mengimpor library yang umum digunakan untuk analisis data dan machine learning. Kita juga menyiapkan konfigurasi umum seperti kolom target, metrik evaluasi, dan style visualisasi.

In [ ]:
# Contoh: muat data dari CSV
# df = pd.read_csv("data/face_dataset.csv")
# print(df.head())
# print(df.info())

# Simulasi dataset contoh jika file belum tersedia
rng = np.random.default_rng(42)
feature_count = 10
n_rows = 200
X = rng.normal(size=(n_rows, feature_count))
# Ciri-ciri yang membedakan dua kelas
X[:n_rows // 2] += 1.5
labels = np.array([0] * (n_rows // 2) + [1] * (n_rows - n_rows // 2))
df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(feature_count)])
df[TARGET_COLUMN] = labels

df.head()

## 2. Muat Dataset

Pada bagian ini kita membaca dataset, melihat preview awal, dan menganalisa struktur data. Informasi seperti jumlah baris, kolom, serta tipe data akan membantu menentukan langkah preprocessing yang perlu dilakukan.

In [ ]:
print(f"Jumlah baris: {len(df)}")
print(f"Jumlah kolom: {len(df.columns)}")
print(df.dtypes.to_string())
print("\nPreview awal:")
df.head()

## 3. Siapkan Data Latih dan Uji

Setelah dataset terbaca, kita memisahkan fitur dan target, lalu membagi data menjadi train set dan test set. Langkah ini penting agar evaluasi model dilakukan pada data yang benar-benar belum pernah dilihat model saat training.

In [ ]:
X = df.drop(columns=[TARGET_COLUMN])
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features)
    ],
    remainder="drop",
)

preprocessor

## 4. Definisikan Model yang Dapat Dipilih

Berikut daftar model yang akan dibandingkan. Setiap model disimpan dalam dictionary agar mudah dipilih saat runtime dan dapat ditambahkan atau dihapus tanpa mengubah logika evaluasi utama.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
}

for name, model in models.items():
    print(f"- {name}: {type(model).__name__}")

## 5. Buat Fungsi Evaluasi Model

Fungsi evaluasi digunakan untuk melatih model, memprediksi target pada test set, dan menghitung beberapa metrik performa. Dengan pendekatan seperti ini, kita dapat membandingkan model dengan cara yang konsisten.

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    pred = pipeline.predict(X_test)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
    }
    if hasattr(pipeline, "predict_proba"):
        proba = pipeline.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, proba)
    return metrics, pred

print("Fungsi evaluasi siap dipakai.")

## 6. Tambahkan Fitur Pilihan Model

Pada bagian ini, kita membuat antarmuka interaktif agar pengguna dapat memilih model yang ingin dibandingkan. Metode yang umum digunakan adalah dropdown atau widget select yang diambil dari daftar model yang tersedia.

In [ ]:
available_models = list(models.keys())
selected_models = ["Logistic Regression", "Random Forest", "SVM"]

print("Model yang tersedia:")
for m in available_models:
    print(f"- {m}")

print("\nModel yang dipilih user secara default:")
for m in selected_models:
    print(f"- {m}")

## 7. Bandingkan Performa Model

Setelah model dipilih, kita iterasi model satu per satu, latih model, evaluasi performa, lalu menyimpan hasilnya ke DataFrame. Hasil ini mempermudah kita melihat model mana yang paling unggul berdasarkan metrik tertentu.

In [ ]:
results = []
for model_name in selected_models:
    model = models[model_name]
    metrics, _ = evaluate_model(model_name, model, X_train, X_test, y_train, y_test)
    results.append(metrics)

comparison_df = pd.DataFrame(results).sort_values(by=METRIC, ascending=False)
comparison_df

## 8. Visualisasikan Hasil Perbandingan

Visualisasi membantu memahami performa setiap model secara cepat. Dalam praktiknya, kita dapat membandingkan metrik seperti akurasi atau F1-score dengan bar chart, serta menampilkan confusion matrix atau ROC curve jika dibutuhkan.

In [ ]:
metric_cols = ["accuracy", "precision", "recall", "f1"]
plot_df = comparison_df.set_index("model")[metric_cols]

plt.figure(figsize=(10, 5))
plot_df.plot(kind="bar", ax=plt.gca())
plt.title("Perbandingan Metrik Model")
plt.ylabel("Skor")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("Model terbaik berdasarkan F1-score:")
print(comparison_df.iloc[0][["model", METRIC]].to_dict())

## 9. Simpan dan Tampilkan Ringkasan

Pada tahap akhir, kita menampilkan ringkasan hasil, menyimpan DataFrame ke file CSV atau Excel, dan merekap model terbaik berdasarkan metrik yang dipilih.

In [ ]:
comparison_df.to_csv("model_comparison_results.csv", index=False)
print("Hasil evaluasi telah disimpan ke model_comparison_results.csv")

best_model = comparison_df.iloc[0]
print("\nRingkasan model terbaik:")
print(best_model.to_dict())

# Notebook: Model Comparison for Face Recognition

Notebook ini berfungsi untuk membandingkan beberapa model dalam proyek machine learning, terutama saat ingin memilih model terbaik untuk klasifikasi atau pengenalan wajah. Fokus utamanya adalah fitur pemilihan model yang dapat dipilih secara interaktif sebelum melakukan evaluasi dan perbandingan performa.